# 03 — Sequence Model
## AT&T Spam Detector

### Objective

This notebook evaluates whether explicitly modeling token order improves SMS spam classification beyond the embedding + global-average-pooling baseline.

The workflow will:

1. load the prepared train and validation splits;
2. recreate the corrected training-only text vectorization pipeline;
3. build a recurrent neural network using a GRU layer;
4. train the model with the same validation safeguards used for the baseline;
5. evaluate precision, recall, F1-score, ROC-AUC, PR-AUC, and classification errors;
6. compare the sequence model against the baseline under the same validation protocol.

The test set remains untouched during model development and is reserved for final evaluation.


In [1]:
# ---------------------------------------------------------------------------
# Imports and reproducibility
# ---------------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

TensorFlow version: 2.21.0
NumPy version: 2.5.2
Pandas version: 3.0.5


### 3.1 Load prepared datasets

The train, validation, and test splits created during preprocessing are reused so that model comparisons are performed on identical observations.

Only the training and validation splits are used during model development. The test split is loaded for completeness but remains untouched until final model evaluation.

In [2]:
# ---------------------------------------------------------------------------
# Locate processed dataset splits
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DATA_DIR / "train.csv"
VAL_PATH = PROCESSED_DATA_DIR / "validation.csv"
TEST_PATH = PROCESSED_DATA_DIR / "test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train shape:      {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape:       {test_df.shape}")

Train shape:      (3610, 2)
Validation shape: (774, 2)
Test shape:       (774, 2)


In [3]:
# ---------------------------------------------------------------------------
# Prepare text inputs and binary labels
# ---------------------------------------------------------------------------

X_train = train_df["text"].copy()
X_val = val_df["text"].copy()
X_test = test_df["text"].copy()

label_mapping = {
    "ham": 0,
    "spam": 1,
}

y_train = train_df["label"].map(label_mapping)
y_val = val_df["label"].map(label_mapping)
y_test = test_df["label"].map(label_mapping)

print("Training class counts:")
print(y_train.value_counts().sort_index())

print("\nValidation class counts:")
print(y_val.value_counts().sort_index())

Training class counts:
label
0    3161
1     449
Name: count, dtype: int64

Validation class counts:
label
0    677
1     97
Name: count, dtype: int64


### 3.2 Recreate the robust text vectorization pipeline

The same training-only preprocessing strategy used by the corrected baseline is reused for the sequence model.

Text is lowercased and punctuation is removed. If a message becomes empty after standardization, it is replaced with a dedicated `[EMPTY]` token so that no observation becomes fully padded.

The vectorizer is adapted exclusively on the training split to prevent vocabulary leakage from validation or test data.

In [4]:
# ---------------------------------------------------------------------------
# Define robust text standardization
# ---------------------------------------------------------------------------

from tensorflow.keras.layers import TextVectorization

MAX_SEQUENCE_LENGTH = 50


def safe_standardize(text):
    """
    Lowercase text, strip punctuation, and replace empty standardized
    messages with a dedicated token.
    """

    # Convert text to lowercase.
    text = tf.strings.lower(text)

    # Remove punctuation while retaining alphanumeric/text content.
    text = tf.strings.regex_replace(
        text,
        r"[!\"#$%&'()*+,-./:;<=>?@\[\\\]^_`{|}~]",
        "",
    )

    # Remove leading and trailing whitespace.
    text = tf.strings.strip(text)

    # Prevent punctuation-only SMS messages from becoming empty sequences.
    text = tf.where(
        tf.strings.length(text) > 0,
        text,
        "[EMPTY]",
    )

    return text

In [5]:
# ---------------------------------------------------------------------------
# Create and adapt the text vectorizer
# ---------------------------------------------------------------------------

text_vectorizer = TextVectorization(
    standardize=safe_standardize,
    split="whitespace",
    output_mode="int",
    output_sequence_length=MAX_SEQUENCE_LENGTH,
)

# Learn vocabulary only from training data.
text_vectorizer.adapt(X_train)

vocabulary = text_vectorizer.get_vocabulary()
VOCAB_SIZE = len(vocabulary)

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Sequence length: {MAX_SEQUENCE_LENGTH}")

Vocabulary size: 7649
Sequence length: 50


In [6]:
# ---------------------------------------------------------------------------
# Convert SMS text into fixed-length integer sequences
# ---------------------------------------------------------------------------

X_train_vectorized = text_vectorizer(X_train)
X_val_vectorized = text_vectorizer(X_val)
X_test_vectorized = text_vectorizer(X_test)

print(f"Training tensor shape:   {X_train_vectorized.shape}")
print(f"Validation tensor shape: {X_val_vectorized.shape}")
print(f"Test tensor shape:       {X_test_vectorized.shape}")

Training tensor shape:   (3610, 50)
Validation tensor shape: (774, 50)
Test tensor shape:       (774, 50)


In [7]:
# ---------------------------------------------------------------------------
# Verify that no SMS becomes a fully padded sequence
# ---------------------------------------------------------------------------

for name, data in {
    "Train": X_train_vectorized,
    "Validation": X_val_vectorized,
    "Test": X_test_vectorized,
}.items():

    fully_padded_count = np.all(
        data.numpy() == 0,
        axis=1,
    ).sum()

    print(
        f"{name:10s} fully padded sequences: "
        f"{fully_padded_count}"
    )

Train      fully padded sequences: 0
Validation fully padded sequences: 0
Test       fully padded sequences: 0


### 3.3 Build a GRU sequence model

The baseline model averages token embeddings and therefore does not explicitly preserve word order.

To test whether sequential context improves classification, a Gated Recurrent Unit (GRU) layer is introduced.

The architecture is:

**Embedding → GRU → Dense → Dropout → Sigmoid**

The GRU processes tokens in sequence and can learn dependencies between words while remaining lighter than a comparable LSTM architecture.

In [8]:
# ---------------------------------------------------------------------------
# Build the GRU sequence model
# ---------------------------------------------------------------------------

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Embedding,
    GRU,
    Dense,
    Dropout,
)

EMBEDDING_DIM = 64
GRU_UNITS = 64

gru_model = Sequential(
    [
        # Learn dense vector representations for vocabulary tokens.
        Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=EMBEDDING_DIM,
            mask_zero=True,
            name="embedding",
        ),

        # Process token embeddings sequentially so that word order
        # and contextual relationships can influence the representation.
        GRU(
            GRU_UNITS,
            name="gru",
        ),

        # Learn nonlinear combinations of the sequence representation.
        Dense(
            32,
            activation="relu",
            name="dense_hidden",
        ),

        # Reduce overfitting during training.
        Dropout(
            0.3,
            name="dropout",
        ),

        # Output the probability that an SMS belongs to the spam class.
        Dense(
            1,
            activation="sigmoid",
            name="spam_probability",
        ),
    ],
    name="gru_sequence_model",
)

# Build the model explicitly so that the architecture and parameter
# counts can be inspected before training.
gru_model.build(
    input_shape=(None, MAX_SEQUENCE_LENGTH)
)

gru_model.summary()

Model: "gru_sequence_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 64)         │       489,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spam_probability (Dense)        │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 516,609 (1.97 MB)

 Trainable params: 516,609 (1.97 MB)

 Non-trainable params: 0 (0.00 B)

### 3.4 Compile the GRU model

The GRU model is compiled using the same optimization objective and evaluation metrics as the baseline.

Using the same binary cross-entropy loss, optimizer family, and classification metrics ensures that performance differences are attributable primarily to the architecture rather than to a different evaluation setup.

In [9]:
# ---------------------------------------------------------------------------
# Compile the GRU sequence model
# ---------------------------------------------------------------------------

from tensorflow.keras.metrics import (
    BinaryAccuracy,
    Precision,
    Recall,
    AUC,
)

gru_model.compile(
    # Adam provides a strong general-purpose optimizer for this experiment.
    optimizer="adam",

    # Binary cross-entropy matches the binary target and sigmoid output.
    loss="binary_crossentropy",

    # Use the same metrics as the baseline for direct comparison.
    metrics=[
        BinaryAccuracy(name="accuracy"),
        Precision(name="precision"),
        Recall(name="recall"),
        AUC(name="roc_auc"),
        AUC(name="pr_auc", curve="PR"),
    ],
)

In [10]:
# ---------------------------------------------------------------------------
# Configure training safeguards
# ---------------------------------------------------------------------------

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
)

early_stopping = EarlyStopping(
    # Monitor generalization through validation loss.
    monitor="val_loss",
    patience=3,

    # Keep the weights from the epoch with the lowest validation loss.
    restore_best_weights=True,
)

reduce_lr = ReduceLROnPlateau(
    # Reduce the learning rate if validation loss stops improving.
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

### 3.5 Train the GRU sequence model

The GRU model is trained using the same batch size, maximum number of epochs, and validation protocol as the baseline.

This controlled setup allows the recurrent architecture to be compared fairly with the simpler global-average-pooling model.

The test set remains untouched.

In [11]:
# ---------------------------------------------------------------------------
# Train the GRU sequence model
# ---------------------------------------------------------------------------

EPOCHS = 20
BATCH_SIZE = 32

history_gru = gru_model.fit(
    # Train on the same vectorized training split as the baseline.
    X_train_vectorized,
    y_train,

    # Monitor performance on the validation split.
    validation_data=(
        X_val_vectorized,
        y_val,
    ),

    # Keep the same training configuration as the baseline.
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,

    # Limit overfitting and reduce the learning rate when needed.
    callbacks=[
        early_stopping,
        reduce_lr,
    ],

    # Print one line of metrics per epoch.
    verbose=1,
)

Epoch 1/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.9102 - loss: 0.2759 - pr_auc: 0.5709 - precision: 0.8415 - recall: 0.3430 - roc_auc: 0.8564 - val_accuracy: 0.9690 - val_loss: 0.0870 - val_pr_auc: 0.9504 - val_precision: 0.8842 - val_recall: 0.8660 - val_roc_auc: 0.9813 - learning_rate: 0.0010
Epoch 2/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9898 - loss: 0.0397 - pr_auc: 0.9845 - precision: 0.9640 - recall: 0.9532 - roc_auc: 0.9934 - val_accuracy: 0.9742 - val_loss: 0.0728 - val_pr_auc: 0.9640 - val_precision: 0.8969 - val_recall: 0.8969 - val_roc_auc: 0.9847 - learning_rate: 0.0010
Epoch 3/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9986 - loss: 0.0105 - pr_auc: 0.9972 - precision: 1.0000 - recall: 0.9889 - roc_auc: 0.9986 - val_accuracy: 0.9767 - val_loss: 0.0815 - val_pr_auc: 0.9631 - val_precision: 0.8990 - val_recall: 0.9175 - val_roc_auc: 0.9810 - learning_rate: 0.0010
Epoch 4/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accura

### 3.6 Evaluate the restored GRU model

The GRU begins to overfit after only a few epochs: training performance continues to improve while validation loss increases.

Because early stopping restores the weights associated with the lowest validation loss, the retained model is evaluated separately after training.

This provides a fair comparison with the baseline model.

In [12]:
# ---------------------------------------------------------------------------
# Evaluate the restored GRU model on validation data
# ---------------------------------------------------------------------------

validation_results_gru = gru_model.evaluate(
    X_val_vectorized,
    y_val,
    verbose=0,
    return_dict=True,
)

print("GRU validation metrics using restored best weights:\n")

for metric_name, metric_value in validation_results_gru.items():
    print(f"{metric_name:10s}: {metric_value:.4f}")

GRU validation metrics using restored best weights:

accuracy  : 0.9742
loss      : 0.0728
pr_auc    : 0.9640
precision : 0.8969
recall    : 0.8969
roc_auc   : 0.9847


In [13]:
# ---------------------------------------------------------------------------
# Generate validation predictions at the default threshold
# ---------------------------------------------------------------------------

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

# Generate continuous spam probabilities.
val_probabilities_gru = gru_model.predict(
    X_val_vectorized,
    verbose=0,
).ravel()

# Apply the standard binary threshold.
val_predictions_gru_05 = (
    val_probabilities_gru >= 0.50
).astype(int)

precision_gru_05 = precision_score(
    y_val,
    val_predictions_gru_05,
)

recall_gru_05 = recall_score(
    y_val,
    val_predictions_gru_05,
)

f1_gru_05 = f1_score(
    y_val,
    val_predictions_gru_05,
)

print(f"Precision: {precision_gru_05:.4f}")
print(f"Recall:    {recall_gru_05:.4f}")
print(f"F1-score:  {f1_gru_05:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_val,
        val_predictions_gru_05,
        target_names=["ham", "spam"],
        digits=4,
    )
)

print("Confusion matrix:")
print(
    confusion_matrix(
        y_val,
        val_predictions_gru_05,
    )
)

Precision: 0.8969
Recall:    0.8969
F1-score:  0.8969

Classification report:
              precision    recall  f1-score   support

         ham     0.9852    0.9852    0.9852       677
        spam     0.8969    0.8969    0.8969        97

    accuracy                         0.9742       774
   macro avg     0.9411    0.9411    0.9411       774
weighted avg     0.9742    0.9742    0.9742       774

Confusion matrix:
[[667  10]
 [ 10  87]]


### 3.8 Investigate the GRU classification threshold

Decision thresholds are model-specific because different architectures can produce differently calibrated probability scores.

The GRU threshold is therefore selected independently using validation data only. The test set remains untouched.

In [14]:
# ---------------------------------------------------------------------------
# Compare GRU candidate thresholds on validation data
# ---------------------------------------------------------------------------

threshold_results_gru = []

candidate_thresholds = np.arange(
    0.10,
    0.91,
    0.05,
)

for threshold in candidate_thresholds:

    predictions = (
        val_probabilities_gru >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions,
    ).ravel()

    threshold_results_gru.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "false_positives": fp,
        "false_negatives": fn,
    })

threshold_df_gru = pd.DataFrame(
    threshold_results_gru
)

print(
    threshold_df_gru.round(4).to_string(
        index=False
    )
)

 threshold  precision  recall     f1  false_positives  false_negatives
      0.10     0.8182  0.9278 0.8696               20                7
      0.15     0.8182  0.9278 0.8696               20                7
      0.20     0.8396  0.9175 0.8768               17                8
      0.25     0.8476  0.9175 0.8812               16                8
      0.30     0.8558  0.9175 0.8856               15                8
      0.35     0.8725  0.9175 0.8945               13                8
      0.40     0.8713  0.9072 0.8889               13                9
      0.45     0.8800  0.9072 0.8934               12                9
      0.50     0.8969  0.8969 0.8969               10               10
      0.55     0.9062  0.8969 0.9016                9               10
      0.60     0.9158  0.8969 0.9062                8               10
      0.65     0.9158  0.8969 0.9062                8               10
      0.70     0.9355  0.8969 0.9158                6               10
      

In [15]:
# ---------------------------------------------------------------------------
# Identify the GRU threshold with the highest validation F1-score
# ---------------------------------------------------------------------------

best_threshold_row_gru = threshold_df_gru.loc[
    threshold_df_gru["f1"].idxmax()
]

print("Best GRU threshold by validation F1-score:")
print(
    best_threshold_row_gru.round(4)
)

Best GRU threshold by validation F1-score:
threshold           0.9000
precision           1.0000
recall              0.8763
f1                  0.9341
false_positives     0.0000
false_negatives    12.0000
Name: 16, dtype: float64


In [ ]:
""""
The GRU does not beat the baseline at the default threshold, but after validation 
threshold selection it reaches a higher F1.

At 0.50, GRU F1 is only 0.8969 versus the baseline's 0.9146. 
At its validation-selected threshold of 0.90, however, GRU F1 rises to 0.9341, 
versus the baseline's selected-threshold F1 of 0.9200.

There is an important trade-off: the GRU at 0.90 produces zero false positives, 
but misses 12 spam messages. The baseline at 0.40 produces 11 false positives, 
but misses only 5 spam messages. So saying simply "GRU is better" would be misleading.

Also, the fact that the best value is at the upper boundary of my search (0.90) means 
I should not freeze 0.90 yet. The true validation F1 optimum could be above it."""

### 3.9 Refine the GRU threshold search

The initial threshold search found its highest validation F1-score at 0.90, which was the upper boundary of the evaluated range.

Because an optimum occurring at the search boundary may indicate that better thresholds exist outside the tested interval, a finer search is performed around the high-probability region.

This refinement uses the validation set only. The test set remains untouched.

In [16]:
# ---------------------------------------------------------------------------
# Refine the GRU threshold search around the high-probability region
# ---------------------------------------------------------------------------

refined_threshold_results_gru = []

# Search more finely from 0.80 through 0.99.
refined_thresholds = np.arange(
    0.80,
    1.00,
    0.01,
)

for threshold in refined_thresholds:

    # Convert GRU probabilities into binary predictions.
    predictions = (
        val_probabilities_gru >= threshold
    ).astype(int)

    # Extract business-relevant error counts.
    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions,
    ).ravel()

    refined_threshold_results_gru.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "false_positives": fp,
        "false_negatives": fn,
    })

refined_threshold_df_gru = pd.DataFrame(
    refined_threshold_results_gru
)

print(
    refined_threshold_df_gru.round(4).to_string(
        index=False
    )
)

 threshold  precision  recall     f1  false_positives  false_negatives
      0.80     0.9773  0.8866 0.9297                2               11
      0.81     0.9773  0.8866 0.9297                2               11
      0.82     0.9773  0.8866 0.9297                2               11
      0.83     0.9773  0.8866 0.9297                2               11
      0.84     0.9773  0.8866 0.9297                2               11
      0.85     0.9773  0.8866 0.9297                2               11
      0.86     0.9773  0.8866 0.9297                2               11
      0.87     0.9773  0.8866 0.9297                2               11
      0.88     0.9885  0.8866 0.9348                1               11
      0.89     1.0000  0.8866 0.9399                0               11
      0.90     1.0000  0.8763 0.9341                0               12
      0.91     1.0000  0.8763 0.9341                0               12
      0.92     1.0000  0.8763 0.9341                0               12
      

In [17]:
# ---------------------------------------------------------------------------
# Identify the best threshold in the refined search
# ---------------------------------------------------------------------------

best_refined_threshold_row_gru = (
    refined_threshold_df_gru.loc[
        refined_threshold_df_gru["f1"].idxmax()
    ]
)

print(
    "Best refined GRU threshold by validation F1-score:"
)

print(
    best_refined_threshold_row_gru.round(4)
)

Best refined GRU threshold by validation F1-score:
threshold           0.8900
precision           1.0000
recall              0.8866
f1                  0.9399
false_positives     0.0000
false_negatives    11.0000
Name: 9, dtype: float64


### 3.10 Compare the GRU with the baseline

The sequence model is compared with the previously established embedding + global-average-pooling baseline.

Each model uses a decision threshold selected exclusively from validation data. This allows the comparison to reflect the best observed validation operating point for each architecture while keeping the test set untouched.

Performance is assessed using precision, recall, F1-score, ROC-AUC, PR-AUC, and the numbers of false-positive and false-negative classifications.

In [18]:
# ---------------------------------------------------------------------------
# Compare validation performance of the baseline and GRU models
# ---------------------------------------------------------------------------

model_comparison = pd.DataFrame(
    [
        {
            "model": "Embedding + GlobalAveragePooling1D",
            "threshold": 0.40,
            "precision": 0.8932,
            "recall": 0.9485,
            "f1": 0.9200,
            "roc_auc": 0.9939,
            "pr_auc": 0.9531,
            "false_positives": 11,
            "false_negatives": 5,
        },
        {
            "model": "Embedding + GRU",
            "threshold": 0.89,
            "precision": 1.0000,
            "recall": 0.8866,
            "f1": 0.9399,
            "roc_auc": validation_results_gru["roc_auc"],
            "pr_auc": validation_results_gru["pr_auc"],
            "false_positives": 0,
            "false_negatives": 11,
        },
    ]
)

model_comparison.round(4)

,model,threshold,precision,recall,f1,roc_auc,pr_auc,false_positives,false_negatives
0,Embedding + GlobalAveragePooling1D,0.40,0.8932,0.9485,0.9200,0.9939,0.9531,11,5
1,Embedding + GRU,0.89,1.0000,0.8866,0.9399,0.9847,0.9640,0,11


#### Interim model comparison

The GRU and baseline exhibit different validation trade-offs.

At their validation-selected thresholds, the GRU achieves the higher F1-score (0.9399 vs. 0.9200) and eliminates false positives on the validation set. Its precision therefore reaches 1.0000. However, this comes at the cost of lower spam recall: the GRU misses 11 spam messages, compared with 5 for the baseline.

The threshold-independent metrics are also mixed. The baseline achieves the higher ROC-AUC (0.9939 vs. 0.9847), while the GRU achieves the higher PR-AUC (0.9640 vs. 0.9531).

Consequently, the GRU cannot yet be considered unambiguously superior. Its sequential architecture produces a more conservative classifier at the selected operating threshold, whereas the simpler baseline detects a larger proportion of spam.

The final model choice will therefore consider both statistical performance and the business consequences of false positives and false negatives.

### 3.11 GRU error analysis

Aggregate metrics do not explain which messages a model fails to classify correctly.

At the validation-selected threshold, the GRU produces no false positives but still misses several spam messages. These false negatives are inspected to determine whether they share linguistic characteristics and whether sequence modeling resolves the errors observed with the baseline.


In [19]:
# ---------------------------------------------------------------------------
# Identify GRU classification errors at the selected threshold
# ---------------------------------------------------------------------------

GRU_THRESHOLD = 0.89

val_predictions_gru = (
    val_probabilities_gru >= GRU_THRESHOLD
).astype(int)

error_analysis_gru = val_df.copy()

error_analysis_gru["true_label"] = y_val.to_numpy()
error_analysis_gru["spam_probability"] = val_probabilities_gru
error_analysis_gru["prediction"] = val_predictions_gru

false_positives_gru = error_analysis_gru[
    (error_analysis_gru["true_label"] == 0)
    & (error_analysis_gru["prediction"] == 1)
].copy()

false_negatives_gru = error_analysis_gru[
    (error_analysis_gru["true_label"] == 1)
    & (error_analysis_gru["prediction"] == 0)
].copy()

print(
    f"False positives: {len(false_positives_gru)}"
)
print(
    f"False negatives: {len(false_negatives_gru)}"
)

False positives: 0
False negatives: 11


In [20]:
# ---------------------------------------------------------------------------
# Inspect spam messages missed by the GRU
# ---------------------------------------------------------------------------

false_negatives_gru[
    [
        "text",
        "spam_probability",
    ]
].sort_values(
    "spam_probability",
    ascending=True,
)

,text,spam_probability
219,dating:i have had two of these. Only started a...,0.000387
468,Dear Voucher Holder 2 claim your 1st class air...,0.002409
108,You'll not rcv any more msgs from the chat svc...,0.005940
411,For sale - arsenal dartboard. Good condition b...,0.017133
383,"Hi this is Amy, we will be sending you a free ...",0.026090
726,Babe: U want me dont u baby! Im nasty and have...,0.046427
101,"Latest News! Police station toilet stolen, cop...",0.068967
433,1000's of girls many local 2 u who r virgins 2...,0.168787
716,Monthly password for wap. mobsi.com is 391784....,0.386492
619,thesmszone.com lets you send free anonymous an...,0.468938


In [ ]:
"""
Several of the GRU's misses are not borderline at all. Messages such as the dating message 
(0.000387), voucher/airfare offer (0.002409), chat-service message (0.005940), 
and sale message (0.017133) are classified with very high confidence as ham. 
So simply lowering the GRU threshold would not recover many of these without potentially 
changing its overall error profile substantially.

Also, several messages that the baseline missed appear again here: the dating message, 
Arsenal dartboard, Amy/free-phone message, and adult-service message. 
That suggests the GRU has not solved the baseline's difficult cases simply by 
modeling word order. Its higher validation F1 comes largely from a different precision/recall
 operating point.

"""

#### GRU error interpretation

At the selected threshold of 0.89, all 11 validation errors are false negatives and no legitimate messages are classified as spam.

Several missed spam messages receive extremely low spam probabilities rather than falling just below the decision threshold. Examples include promotional, subscription, dating, and sales-oriented messages. This indicates that the remaining errors cannot be explained solely by the relatively high classification threshold.

Several difficult messages are also similar to, or overlap with, errors observed for the baseline model. Therefore, explicitly modeling token order with a GRU does not consistently resolve the challenging cases identified by the simpler architecture.

The GRU nevertheless changes the classifier's operating characteristics substantially. At its validation-selected threshold it achieves higher F1 and perfect validation precision, but lower spam recall than the baseline. This makes the GRU more conservative: it is less likely to incorrectly block legitimate SMS messages, but more likely to allow spam through.

These results reinforce the need to compare models using both aggregate metrics and the business consequences of false positives and false negatives rather than selecting a model from accuracy alone.

In [21]:
# ---------------------------------------------------------------------------
# Save GRU validation metrics for later model comparison
# ---------------------------------------------------------------------------

gru_validation_metrics = pd.DataFrame(
    [
        {
            "model": "Embedding + GRU",
            "selected_threshold": GRU_THRESHOLD,
            "validation_accuracy": validation_results_gru["accuracy"],
            "validation_precision": 1.0000,
            "validation_recall": 0.8866,
            "validation_f1": 0.9399,
            "validation_roc_auc": validation_results_gru["roc_auc"],
            "validation_pr_auc": validation_results_gru["pr_auc"],
            "validation_false_positives": 0,
            "validation_false_negatives": 11,
        }
    ]
)

METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

GRU_METRICS_PATH = (
    METRICS_DIR
    / "gru_validation_metrics.csv"
)

gru_validation_metrics.to_csv(
    GRU_METRICS_PATH,
    index=False,
)

print(f"Metrics saved to: {GRU_METRICS_PATH}")

Metrics saved to: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\4_Att-spam-detector\outputs\metrics\gru_validation_metrics.csv


In [22]:
# ---------------------------------------------------------------------------
# Save the trained GRU model
# ---------------------------------------------------------------------------

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

GRU_MODEL_PATH = (
    MODELS_DIR
    / "gru_sequence_model.keras"
)

gru_model.save(GRU_MODEL_PATH)

print(f"Model saved to: {GRU_MODEL_PATH}")

Model saved to: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\4_Att-spam-detector\models\gru_sequence_model.keras


In [23]:
# ---------------------------------------------------------------------------
# Verify final GRU metrics at the frozen validation-selected threshold
# ---------------------------------------------------------------------------

final_gru_predictions = (
    val_probabilities_gru >= GRU_THRESHOLD
).astype(int)

final_gru_precision = precision_score(
    y_val,
    final_gru_predictions,
    zero_division=0,
)

final_gru_recall = recall_score(
    y_val,
    final_gru_predictions,
    zero_division=0,
)

final_gru_f1 = f1_score(
    y_val,
    final_gru_predictions,
    zero_division=0,
)

final_gru_confusion_matrix = confusion_matrix(
    y_val,
    final_gru_predictions,
)

print(f"Frozen threshold: {GRU_THRESHOLD:.2f}")
print(f"Precision:        {final_gru_precision:.4f}")
print(f"Recall:           {final_gru_recall:.4f}")
print(f"F1-score:         {final_gru_f1:.4f}")

print("\nConfusion matrix:")
print(final_gru_confusion_matrix)

Frozen threshold: 0.89
Precision:        1.0000
Recall:           0.8866
F1-score:         0.9399

Confusion matrix:
[[677   0]
 [ 11  86]]


## 3.12 Conclusion

A GRU-based sequence model was trained to determine whether explicitly modeling token order improves SMS spam classification beyond the simpler embedding + global-average-pooling baseline.

The GRU reached its lowest validation loss at epoch 2. Training performance continued to improve afterward while validation loss increased, indicating rapid overfitting. Early stopping therefore restored the epoch-2 weights.

At the default threshold of 0.50, the GRU achieved:

- validation accuracy: **0.9742**
- precision: **0.8969**
- recall: **0.8969**
- F1-score: **0.8969**
- ROC-AUC: **0.9847**
- PR-AUC: **0.9640**

Threshold selection was performed using validation data only. Because the initial search reached its optimum near the upper search boundary, the high-probability region was examined more finely. The final validation-selected threshold was **0.89**.

At this threshold, the GRU achieved:

- precision: **1.0000**
- recall: **0.8866**
- F1-score: **0.9399**
- false positives: **0**
- false negatives: **11**

Compared with the baseline at its validation-selected threshold of 0.40, the GRU obtains a higher validation F1-score and eliminates false positives. However, the baseline retains substantially higher spam recall and misses only 5 spam messages instead of 11.

The threshold-independent metrics also provide mixed evidence: the baseline achieves the higher ROC-AUC, whereas the GRU achieves the higher PR-AUC.

Error analysis shows that several GRU false negatives receive very low spam probabilities and that some difficult messages overlap with errors observed for the baseline. Therefore, adding sequential modeling does not consistently resolve the challenging SMS examples.

Overall, the GRU provides a useful alternative operating profile rather than an unambiguously superior model. It is more conservative at its selected threshold, reducing the risk of incorrectly blocking legitimate messages while allowing more spam messages through.

The test set remains untouched for final model evaluation. The next modeling stage will investigate transfer learning before selecting the final candidate architecture.

In [ ]:
"""
One correction before committing: in the earlier metrics-saving cell, 
I hard-coded rounded values. That's acceptable for display, but for a professional 
reproducible pipeline I should save the values actually calculated by the notebook.

"""

In [24]:
# ---------------------------------------------------------------------------
# Save GRU validation metrics for later model comparison
# ---------------------------------------------------------------------------

gru_validation_metrics = pd.DataFrame(
    [
        {
            "model": "Embedding + GRU",
            "selected_threshold": GRU_THRESHOLD,
            "validation_accuracy": validation_results_gru["accuracy"],
            "validation_precision": final_gru_precision,
            "validation_recall": final_gru_recall,
            "validation_f1": final_gru_f1,
            "validation_roc_auc": validation_results_gru["roc_auc"],
            "validation_pr_auc": validation_results_gru["pr_auc"],
            "validation_false_positives": int(
                final_gru_confusion_matrix[0, 1]
            ),
            "validation_false_negatives": int(
                final_gru_confusion_matrix[1, 0]
            ),
        }
    ]
)

METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

GRU_METRICS_PATH = (
    METRICS_DIR
    / "gru_validation_metrics.csv"
)

gru_validation_metrics.to_csv(
    GRU_METRICS_PATH,
    index=False,
)

print(gru_validation_metrics.round(4))
print(f"\nMetrics saved to: {GRU_METRICS_PATH}")

             model  selected_threshold  validation_accuracy  \
0  Embedding + GRU                0.89               0.9742   

   validation_precision  validation_recall  validation_f1  validation_roc_auc  \
0                   1.0             0.8866         0.9399              0.9847   

   validation_pr_auc  validation_false_positives  validation_false_negatives  
0              0.964                           0                          11  

Metrics saved to: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\4_Att-spam-detector\outputs\metrics\gru_validation_metrics.csv
